# Association-mining method comparison

Compare transaction encoding and frequent-itemset implementations on a shared store-period dataset.


## Imports and optional engines


In [ ]:
import os
import pyodbc
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv
from itertools import combinations
from fpgrowth_py import fpgrowth as fpgrowth_py
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import association_rules, fpgrowth as mlxtend_fpgrowth
from pyspark.ml.fpm import FPGrowth
from pyspark.sql import Row, SparkSession
from sqlalchemy.engine import URL
from sqlalchemy import create_engine, text


## Configure paths and the SQL Server source


In [ ]:
# Load environment variables from the project root when available.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

datasets_dir = project_root / "datasets"
env_path = project_root / ".env"
if env_path.exists():
    load_dotenv(env_path)
else:
    load_dotenv()

dw_user = (os.getenv("DATAWAREHOUSE_USER") or "").strip()
dw_password = (os.getenv("DATAWAREHOUSE_PASSWORD") or "").strip()
dw_host = (os.getenv("DATAWAREHOUSE_HOST") or "").strip()
dw_database = (os.getenv("DATAWAREHOUSE_DATABASE") or "").strip()

if not all([dw_user, dw_password, dw_host, dw_database]):
    raise ValueError("Missing one or more DATAWAREHOUSE_* environment variables.")

drivers = pyodbc.drivers()
print("Available ODBC drivers:", drivers)

preferred_driver_names = [
    (os.getenv("SQLSERVER_DRIVER") or "").strip(),
    "ODBC Driver 18 for SQL Server",
    "ODBC Driver 17 for SQL Server",
    "SQL Server",
]

driver = next(
    (
        candidate
        for candidate in preferred_driver_names
        if candidate and any(candidate.lower() in d.lower() for d in drivers)
    ),
    None,
)

if not driver:
    raise RuntimeError(
        f"No supported SQL Server ODBC driver found. Available drivers: {drivers}. "
        "Install Microsoft ODBC Driver 18 for SQL Server and verify the 64-bit ODBC Administrator."
    )

print(f"Using SQL Server driver: {driver}")

connection_string = URL.create(
    drivername="mssql+pyodbc",
    username=dw_user,
    password=dw_password,
    host=dw_host,
    database=dw_database,
    query={
        "driver": driver,
        "Encrypt": "yes",
        "TrustServerCertificate": "yes",
        "LoginTimeout": "30",
    },
)

engine = create_engine(connection_string, pool_pre_ping=True, future=True)

with engine.connect() as conn:
    print("Database connection OK:", conn.execute(text("SELECT 1")).scalar())

query = text(
    """
    SELECT 
        dd.[date] AS TRX_date,
        [StoreCode],
        [BillNo],
        [ItemCode],
        ITEMLONGNAME,
        [Quantity],
        DEPARTMENT,
        CLASS,
        SUBCLASS,
        [UOM_CD],
        [TotalAmt],
        [NetValue],
        [WACValue],
        [CSM_QTY],
        [CONSIGN_FINAL_QTY],
        [POS_FINAL_QTY]
    FROM [DBWH_8555].[dbo].[FactSalesTrxNew] fstn
    INNER JOIN dimdate dd ON dd.datekey = fstn.DateKey
    INNER JOIN DimItem di ON fstn.ItemCode = di.ITMCD
    WHERE dd.[date] BETWEEN :start_date AND :end_date AND StoreCode = '083'
    """)


## Load and validate source transactions


In [ ]:
with engine.connect() as conn:
    start_date = '2025-11-01'
    end_date = '2025-11-30'

    store_df = pd.read_sql(query, conn,  params={'start_date': start_date, 'end_date': end_date})


#df = pd.read_csv('t10_trx_data.csv')
    
df = store_df[~store_df['ITEMLONGNAME'].str.contains('pepito bag', case=False, na=False)].copy()
#df['month_year'] = pd.to_datetime(df['month_year'], format='%m-%Y')
#df['rule'] = df['antecedents'] + " → " + df['consequents']

df.info()
df.head(1)

In [ ]:
print(df.BillNo.nunique())
print(df.ITEMLONGNAME.nunique())

## Select frequent products


In [ ]:
# Get top 10 most frequent products for this store
product_frequency = df.groupby('ITEMLONGNAME').size().sort_values(ascending=False)
top_products = product_frequency.head(10).index

store_df_filtered = df[df['ITEMLONGNAME'].isin(top_products)]

group_item = store_df_filtered.groupby('ITEMLONGNAME').size().sort_values(ascending=False) # calculate the appearance of the top 10 item for each transaction

group_item

## Build binary transaction baskets


In [ ]:
transactions = store_df_filtered.groupby(['BillNo', 'ITEMLONGNAME'])['POS_FINAL_QTY'].sum().unstack(fill_value=0)

transactions = transactions.apply(pd.to_numeric, errors='coerce').fillna(0)
transactions_binary = transactions.gt(0)  # Convert to boolean (True/False)

transactions_binary.info()

In [ ]:
# new method to do transaction binary
transactions_test = df.groupby("BillNo")["ITEMLONGNAME"].apply(list).tolist()

te = TransactionEncoder()
te_ary = te.fit(transactions_test).transform(transactions_test, sparse=True)

transactions_binary_test = pd.DataFrame.sparse.from_spmatrix(
    te_ary,
    columns=te.columns_
).astype(pd.SparseDtype(bool, fill_value=False))
transactions_binary_test.info()
#print(transactions_binary_test.sparse.density)

In [ ]:
transactions_binary_test.head(20)
# transactions_binary_test.to_csv(datasets_dir / "transactions_binary_test.csv", index=False)

In [ ]:
print(transactions_binary_test.head(100))

t10 = transactions_binary_test.head(10)

#convert to csv
# t10.to_csv(datasets_dir / "t10_transactions_binary.csv", index=False)

print(t10)


## Mine itemsets with mlxtend FP-Growth

After creating the binary transaction matrix, calculate support for each itemset. Support is the share of transactions containing the itemset and ranges from 0 to 1.


In [ ]:
frequent_itemsets_test = mlxtend_fpgrowth(
    transactions_binary_test,
    min_support=0.001, # 0.001 = Only include itemsets appearing in ≥ 0.1% of transactions
    use_colnames=True,
    #max_len=2
)

frequent_itemsets_test.tail()


## Generate mlxtend association rules


In [ ]:
rules = association_rules(
    frequent_itemsets_test,
    metric="confidence",
    min_threshold=0.0 # 0.0 for no filter
)

print()
print(rules.info())

## Compare with Spark FP-Growth


In [ ]:
spark = SparkSession.builder.appName("FPGrowthExample").getOrCreate()
transactions = df.groupby("BillNo")["ItemCode"].apply(list).tolist()
df_spark = spark.createDataFrame([Row(items=items) for items in transactions])

fpGrowth = FPGrowth(itemsCol="items", minSupport=0.001, minConfidence=0.3)
# Deduplicate item codes within each BillNo basket; Spark requires unique items per transaction
transactions = (
    df.drop_duplicates(subset=["BillNo", "ItemCode"])
      .groupby("BillNo")["ItemCode"]
      .agg(lambda xs: list(dict.fromkeys(xs.tolist())))
      .tolist()
)

transactions = (
    df.assign(ItemCode=df["ItemCode"].astype(str))
      .drop_duplicates(subset=["BillNo", "ItemCode"])
      .groupby("BillNo")["ItemCode"]
      .agg(lambda xs: list(dict.fromkeys(xs.tolist())))
      .tolist()
)

df_spark = spark.createDataFrame(
    [Row(items=[str(item) for item in items]) for items in transactions if items],
    schema="items array<string>"
)

fpGrowth = FPGrowth(itemsCol="items", minSupport=0.001, minConfidence=0.3)
model = fpGrowth.fit(df_spark)

# 5. Frequent itemsets
model.freqItemsets.show(truncate=False)

# 6. Association rules
model.associationRules.show(truncate=False)

# 7. Prediksi rekomendasi
model.transform(df_spark).show(truncate=False)

## Compare with fpgrowth_py


In [ ]:
#3
transactions = df.groupby("BillNo")["ItemCode"].apply(list).tolist()
n_transactions = len(transactions)

# De-duplicate items inside each basket; fpgrowth_py expects unique items per transaction.
transactions = [list(dict.fromkeys(txn)) for txn in transactions if txn]

# Handle the library returning no data gracefully.
result = fpgrowth_py(transactions, minSupRatio=0.0, minConf=0.0)

if result is None:
    freqItemSet, rules_fpg = [], []
elif isinstance(result, tuple) and len(result) == 2:
    freqItemSet, rules_fpg = result
else:
    freqItemSet, rules_fpg = [], []

transactions_sets = [set(t) for t in transactions]  # convert once, reuse

def calc_support(itemset):
    return sum(1 for t in transactions_sets if itemset.issubset(t)) / n_transactions

frequent_itemsets = pd.DataFrame([
    {"itemsets": frozenset(itemset), "support": calc_support(itemset)}
    for itemset in freqItemSet
])

# Ensure column dtypes
frequent_itemsets["itemsets"] = frequent_itemsets["itemsets"].apply(frozenset)
frequent_itemsets = frequent_itemsets.drop_duplicates(subset=["itemsets"]).reset_index(drop=True)

# --- Ensure subsets (fix for missing singletons) ---
existing = set(frequent_itemsets["itemsets"])
new_rows = []

for itemset in list(existing):
    if len(itemset) > 1:  # only expand if size >= 2
        for i in range(1, len(itemset)):
            for subset in combinations(itemset, i):
                subset_fs = frozenset(subset)
                if subset_fs not in existing:
                    new_rows.append({
                        "itemsets": subset_fs,
                        "support": calc_support(subset_fs)
                    })
                    existing.add(subset_fs)

if new_rows:
    frequent_itemsets = pd.concat(
        [frequent_itemsets, pd.DataFrame(new_rows)],
        ignore_index=True
    )
# -- END --

In [ ]:
rules = association_rules(
            frequent_itemsets, 
            metric="lift", 
            min_threshold=1.0 # lift, 1.0
        )

## Validate support calculations on a sample


In [ ]:
# only 2 itemset with 2 item each pair, posible apriori rules = 4
# find the support for itemset and support for indivicual item in pairset

pair_set = [
        ['AQUA MINERAL WATER 1500ML', 'AQUA MINERAL WATER 600ML'],
        ['FRUIT CUT 30K', 'SUNPRIDE PISANG CAVENDISH'],
        ['AQUA MINERAL WATER 1500ML'], 
        ['AQUA MINERAL WATER 600ML'],
        ['FRUIT CUT 30K'],
        ['SUNPRIDE PISANG CAVENDISH'],
    ]

print(f"Transactions total: {len(t10)}")
print(f"Columns in binary: {t10.columns.nunique()} items")

def find_support(itemset):
    """Compute support of a given itemset from transactions binary df"""
    if all(col in t10.columns for col in itemset):
        count = t10[itemset].all(axis=1).sum()
        support = count / len(t10)
        return count, support
    else:
        return None, None

# loop through all sets
for pair in pair_set:
    count, support = find_support(pair)
    if count is not None:
        print(f"{pair}: support = {count} / {len(t10)} = {support:.6f}")
    else:
        print(f"❌ {pair} not found in transactions_binary.")

In [ ]:
s_bill_data = (t10.index.tolist())
s_bill_data

In [ ]:
small_data = df[df["BillNo"].isin(s_bill_data)]
small_data = small_data.sort_values(by="BillNo", ascending=True).reset_index(drop=True)

print(small_data.columns)



## Export reproducible sample datasets


In [ ]:
small_data = small_data[
            ['TRX_date', 'BillNo', 'ItemCode', 'ITEMLONGNAME', 'Quantity','UOM_CD', 'TotalAmt',
            'NetValue', 'POS_FINAL_QTY']
        ]

small_data.to_csv(datasets_dir / "t10_trx_data.csv")
small_data
